# EEG 跨数据集一致性分析（通用重构版）

读取**任意通道数、任意窗口长度**的新版或旧版 NPZ，自动输出每文件 QC、跨数据集 PSD / 频带功率 / FC / 通道重叠、JS 散度及 HTML 图册。此 Notebook 仅做**描述性分析**，不会将不同记录自动视为独立被试。

首次运行：在项目根目录执行 `pip install -r requirements.txt`。可先运行 `python examples/create_demo.py` 生成合成演示数据。

In [ ]:
from pathlib import Path
from eeg_visualization import main, read_npz, describe
import pandas as pd

INPUT_DIR = Path('examples/demo_npz')  # 改成自己的 NPZ 文件夹
OUTPUT_DIR = Path('reports/notebook_consistency')
MAX_FILES = 40
MAX_WINDOWS = 64
print('Input:', INPUT_DIR.resolve())

In [ ]:
files = sorted(INPUT_DIR.rglob('*.npz')) if INPUT_DIR.exists() else []
print('NPZ files found:', len(files))
if not files:
    print('提示：先运行 python examples/create_demo.py，或修改 INPUT_DIR')

In [ ]:
if files:
    main(['--input-dir',str(INPUT_DIR), '--output-dir',str(OUTPUT_DIR),
          '--plots','default', '--max-files',str(MAX_FILES),
          '--max-windows',str(MAX_WINDOWS), '--compare', '--dpi','150'])
    display(pd.read_csv(OUTPUT_DIR/'dataset_summary.csv'))

In [ ]:
if files:
    display(pd.read_csv(OUTPUT_DIR/'channel_overlap.csv'))
    display(pd.read_csv(OUTPUT_DIR/'psd_js_divergence.csv'))
    display(pd.read_csv(OUTPUT_DIR/'fc_validation.csv'))

In [ ]:
if files:
    from IPython.display import HTML
    display(HTML(f'<p>已生成 HTML 图册：<code>{OUTPUT_DIR.resolve() / "index.html"}</code></p>'))
    for img in ['compare_rms.png','compare_psd.png','compare_bandpower.png',
                'compare_common_fc.png','compare_channel_overlap.png','compare_psd_js.png']:
        if (OUTPUT_DIR/img).exists():
            display(__import__('IPython').display.Image(filename=str(OUTPUT_DIR/img)))

## 方法说明

- PSD 由 Welch 方法估计；以文件为统计单位做数据集均值。
- JS divergence 计算 1 Hz 到所有数据集共同 Nyquist / 45 Hz 上的归一化 PSD，范围 0–1 bits；**不是统计显著性结果**。
- 比较 FC 仅取**完全相同名称**的电极通道交集，且文件级 FC 均值不会自动校正站点效应。
- 通道重叠率为 Jaccard（通道名称交集/并集），不是空间坐标的等价保证。
- 头皮图基于存储的三维传感器坐标；旧文件缺失坐标时尝试 MNE 标准 montage **近似位置**，实际设备位置需单独核验。
- `max_windows` 抽样是为了降低可视化计算量，不代表整个 NPZ 的质量分布。